# Experimento 2 — Fine-tuning completo do Pollen AI Atlas nos folds da tese
**Pergunta:** treinando o ViT inteiro com o pólen (como a DenseNet201 da tese foi treinada), o Atlas alcança ou supera a DenseNet201 nas vistas equatorial e polar?

**Protocolo:** os mesmos 10 folds e a mesma separação por vistas da tese (`vistas_folds_julio_CPD1.csv`). Em cada fold, 10% do treino vira validação para escolher a melhor época; o teste só é usado no final.

**Aumento de dados e balanceamento** são feitos durante o treino (rotação, translação, espelhamento, desfoque, brilho e contraste, como no código do Júlio), com sorteio balanceado entre as espécies. As imagens `aug_` da pasta **não** são usadas: elas vieram dos seus folds, e não dos folds da tese.

**Os resultados são salvos fold a fold no Drive.** Se o Colab desconectar ou acabar o tempo de GPU, rode tudo de novo noutro dia: os folds já concluídos são pulados.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente → **GPU T4**.

In [ ]:
!pip install -q timm openpyxl
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copia a base para o disco do Colab (precisa repetir a cada nova sessão)
!cp -n /content/drive/MyDrive/IC_Polen/CPD1_Cr_Rs_400.zip /content/
!unzip -q -n /content/CPD1_Cr_Rs_400.zip -d /content/BD/
!ls /content/BD/CPD1_Cr_Rs_400

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
balance_log_2026-11-10.csv  csv  Test  Train


## 1. Configuração — ajuste só esta célula
Comece com `FOLDS = [1]` e me conte quanto tempo o fold levou. Depois troque para `list(range(1, 11))`.

In [19]:
DRIVE = "/content/drive/MyDrive/IC_Polen"
PASTA_IMAGENS = "/content/BD/CPD1_Cr_Rs_400/Test"
MAPA = f"{DRIVE}/vistas_folds_julio_CPD1.csv"
BASELINE = f"{DRIVE}/baseline_DenseNet201_tese_por_fold.xlsx"
CKPT = f"{DRIVE}/weights_vit_small_lvd_20250620_0312.pth"
LP_RESULTADOS = f"{DRIVE}/resultados_atlas_v2/metricas_por_fold.csv"   # Experimento 1 (linear probe)
OUT = f"{DRIVE}/resultados_finetuning"

#MODELOS_FT = ["Atlas"]      # para comparar também o DINOv2 genérico: ["Atlas", "DINOv2"] (dobra o tempo)
MODELOS_FT = ["Atlas", "DINOv2"]
#FOLDS = [1]                 # depois: list(range(1, 11))
FOLDS = list(range(1, 11))

IMG_SIZE = 224              # as imagens do CPD1 já estão em 224x224
EPOCHS = 20
PACIENCIA = 5               # para de treinar se a validação não melhorar por 5 épocas
BATCH = 32
LR_BACKBONE = 2e-5          # taxa pequena para não "esquecer" o pré-treino
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.05
VAL_FRAC = 0.1
SEED = 42

In [20]:
import os, time, numpy as np, pandas as pd
EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
caminho = {}
for raiz, _, arqs in os.walk(PASTA_IMAGENS):
    for a in arqs:
        if a.lower().endswith(EXTS) and not a.lower().startswith("aug_"):
            caminho[f"{os.path.basename(raiz)}/{a}"] = os.path.join(raiz, a)
mapa = pd.read_csv(MAPA)
faltando = sorted(set(mapa.file) - set(caminho))
print(f"Imagens encontradas: {len(caminho)} | na lista da tese: {len(mapa)} | faltando: {len(faltando)}")
classes = sorted(mapa.classe.unique()); idx = {c: i for i, c in enumerate(classes)}
os.makedirs(OUT, exist_ok=True)
print("Checkpoint:", "OK" if os.path.isfile(CKPT) else "NÃO ENCONTRADO", "| Baseline:", "OK" if os.path.isfile(BASELINE) else "NÃO ENCONTRADO")

Imagens encontradas: 4025 | na lista da tese: 4025 | faltando: 0
Checkpoint: OK | Baseline: OK


## 2. Modelo, dados e funções de treino

In [21]:
import torch, timm
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(sem GPU!)")
USA_GPU = device == "cuda"

def criar_modelo(nome):
    m = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=(nome == "DINOv2"),
                          img_size=518, init_values=1e-5, num_classes=0, dynamic_img_size=True)
    if nome == "Atlas":
        sd = torch.load(CKPT, map_location="cpu")
        sd = {k: v for k, v in sd.items() if not k.startswith("head")}
        r = m.load_state_dict(sd, strict=False)
        assert not r.missing_keys, f"Checkpoint não bateu: {r.missing_keys[:5]}"
    m.reset_classifier(len(classes))          # nova camada de saída com as 20 espécies do CPD1
    return m

NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_treino = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=25, translate=(0.2, 0.2)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 0.3))], p=0.3),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.3, contrast=0.25)], p=0.3),
    transforms.ToTensor(), NORM])
tf_teste = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), NORM])

class Polen(Dataset):
    def __init__(self, arquivos, rotulos, tf):
        self.arquivos, self.rotulos, self.tf = list(arquivos), np.asarray(rotulos), tf
    def __len__(self): return len(self.arquivos)
    def __getitem__(self, i):
        return self.tf(Image.open(caminho[self.arquivos[i]]).convert("RGB")), int(self.rotulos[i])

@torch.no_grad()
def prever(modelo, dl):
    modelo.eval(); ps, ys = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USA_GPU):
            ps.append(modelo(x.to(device)).argmax(1).cpu())
        ys.append(y)
    p, y = torch.cat(ps).numpy(), torch.cat(ys).numpy()
    return p, (p == y).mean()

def metricas(yt, yp):
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(yt, yp), "precision": p, "recall": r, "fscore": f,
            "kappa": cohen_kappa_score(yt, yp)}

def treinar_fold(nome, k):
    torch.manual_seed(SEED + k); np.random.seed(SEED + k)
    tr_all, te = mapa[mapa.k != k], mapa[mapa.k == k]
    tr, va = train_test_split(tr_all, test_size=VAL_FRAC, stratify=tr_all.classe, random_state=SEED + k)
    ytr = tr.classe.map(idx).values
    pesos = 1.0 / np.bincount(ytr, minlength=len(classes))[ytr]          # espécies raras sorteadas mais vezes
    sampler = WeightedRandomSampler(torch.as_tensor(pesos, dtype=torch.double), num_samples=len(tr), replacement=True)
    dl_tr = DataLoader(Polen(tr.file, ytr, tf_treino), batch_size=BATCH, sampler=sampler,
                       num_workers=2, pin_memory=True, drop_last=True)
    dl_va = DataLoader(Polen(va.file, va.classe.map(idx), tf_teste), batch_size=64, num_workers=2)
    dl_te = DataLoader(Polen(te.file, te.classe.map(idx), tf_teste), batch_size=64, num_workers=2)

    modelo = criar_modelo(nome).to(device)
    grupos = [{"params": [p for n, p in modelo.named_parameters() if not n.startswith("head")], "lr": LR_BACKBONE},
              {"params": modelo.head.parameters(), "lr": LR_HEAD}]
    opt = torch.optim.AdamW(grupos, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD],
                                                total_steps=EPOCHS * len(dl_tr), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=USA_GPU)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    melhor, estado, sem_melhora, t0 = -1.0, None, 0, time.time()
    for ep in range(1, EPOCHS + 1):
        modelo.train(); perda = 0.0
        for x, y in dl_tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USA_GPU):
                loss = crit(modelo(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            perda += loss.item()
        _, acc_va = prever(modelo, dl_va)
        print(f"  {nome} k{k} época {ep:2d} | perda {perda/len(dl_tr):.4f} | validação {acc_va:.4f} | {(time.time()-t0)/60:.1f} min")
        if acc_va > melhor:
            melhor, sem_melhora = acc_va, 0
            estado = {n: v.detach().cpu().clone() for n, v in modelo.state_dict().items()}
        else:
            sem_melhora += 1
            if sem_melhora >= PACIENCIA:
                print("  parada antecipada"); break
    modelo.load_state_dict(estado)
    yp, _ = prever(modelo, dl_te)
    yte = te.classe.map(idx).values
    del modelo, opt; torch.cuda.empty_cache()
    return te, yte, yp, (time.time() - t0) / 60

Dispositivo: cuda Tesla T4


## 3. Treinar os folds
Cada fold concluído é salvo no Drive na hora. Folds já feitos são pulados.

In [22]:
arq_m, arq_p = f"{OUT}/metricas_por_fold.csv", f"{OUT}/predicoes.csv"
feitos = set()
if os.path.exists(arq_m):
    feitos = {(r.Modelo, r.k) for r in pd.read_csv(arq_m).itertuples()}
for nome in MODELOS_FT:
    for k in FOLDS:
        if (nome, k) in feitos:
            print(f"{nome} k{k}: já feito, pulando"); continue
        print(f"=== {nome} — fold {k} ===")
        te, yte, yp, minutos = treinar_fold(nome, k)
        linhas = []
        for vista in ["TODAS", "EQUATORIAL", "POLAR"]:
            sel = np.ones(len(te), bool) if vista == "TODAS" else (te.vista.values == vista)
            linhas.append({"Modelo": nome, "Vista": vista, "k": k, **metricas(yte[sel], yp[sel]), "minutos": round(minutos, 1)})
        pd.DataFrame(linhas).to_csv(arq_m, mode="a", header=not os.path.exists(arq_m), index=False)
        te.assign(Modelo=nome, predicted_label=[classes[i] for i in yp]) \
          .to_csv(arq_p, mode="a", header=not os.path.exists(arq_p), index=False)
        print(f"{nome} k{k}: todas={linhas[0]['accuracy']:.4f}  equatorial={linhas[1]['accuracy']:.4f}"
              f"  polar={linhas[2]['accuracy']:.4f}  ({minutos:.1f} min)")

Atlas k1: já feito, pulando
Atlas k2: já feito, pulando
Atlas k3: já feito, pulando
Atlas k4: já feito, pulando
Atlas k5: já feito, pulando
Atlas k6: já feito, pulando
Atlas k7: já feito, pulando
Atlas k8: já feito, pulando
Atlas k9: já feito, pulando
Atlas k10: já feito, pulando
DINOv2 k1: já feito, pulando
DINOv2 k2: já feito, pulando
DINOv2 k3: já feito, pulando
DINOv2 k4: já feito, pulando
DINOv2 k5: já feito, pulando
DINOv2 k6: já feito, pulando
DINOv2 k7: já feito, pulando
DINOv2 k8: já feito, pulando
DINOv2 k9: já feito, pulando
DINOv2 k10: já feito, pulando


## 4. Comparar: fine-tuning × linear probe × DenseNet201 da tese

In [23]:
ft = pd.read_csv(arq_m)
ft["Modelo"] = ft.Modelo + " (fine-tuning)"
partes = [ft]
if os.path.exists(LP_RESULTADOS):
    lp = pd.read_csv(LP_RESULTADOS); lp["Modelo"] = lp.Modelo + " (linear probe)"; partes.append(lp)
for vista in ["EQUATORIAL", "POLAR"]:
    d = pd.read_excel(BASELINE, sheet_name=vista)
    d = d[pd.to_numeric(d.k, errors="coerce").notna()].astype({"k": int})
    partes.append(d.assign(Modelo="DenseNet201 (tese)", Vista=vista))
tudo = pd.concat(partes)
tudo = tudo[tudo.k.isin(sorted(ft.k.unique()))]          # compara só os folds já treinados
resumo = (tudo.groupby(["Vista", "Modelo"])[["accuracy", "precision", "recall", "fscore", "kappa"]]
            .agg(["mean", "std"]) * 100).round(2)
print("Folds incluídos:", sorted(ft.k.unique()))
display(resumo)
resumo.to_excel(f"{OUT}/resumo_comparacao.xlsx")

Folds incluídos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


accuracy       precision       recall        \
                                     mean   std      mean   std   mean   std   
Vista      Modelo                                                              
EQUATORIAL Atlas (fine-tuning)      98.76  0.85     98.95  0.81  98.76  0.85   
           Atlas (linear probe)     96.44  0.89     96.91  0.90  96.44  0.89   
           DINOv2 (fine-tuning)     98.33  1.39     98.70  1.25  98.33  1.39   
           DINOv2 (linear probe)    96.44  1.42     96.92  1.45  96.44  1.42   
           DenseNet201 (tese)       98.94  1.17     99.15  0.94  98.94  1.17   
POLAR      Atlas (fine-tuning)      99.38  0.48     99.44  0.48  99.38  0.48   
           Atlas (linear probe)     97.45  1.14     97.85  1.09  97.45  1.14   
           DINOv2 (fine-tuning)     98.95  0.97     99.11  0.84  98.95  0.97   
           DINOv2 (linear probe)    97.92  1.03     98.36  1.06  97.92  1.03   
           DenseNet201 (tese)       99.18  0.79     99.24  0.86  99.18  0.79   
TODAS      Atlas (fine-tuning)      99.06  0.57     99.11  0.55  99.06  0.57   
           Atlas (linear probe)     96.92  0.82     97.08  0.76  96.92  0.82   
           DINOv2 (fine-tuning)     98.63  0.96     98.75  0.85  98.63  0.96   
           DINOv2 (linear probe)    97.14  1.06     97.31  1.02  97.14  1.06   

                                 fscore        kappa        
                                   mean   std   mean   std  
Vista      Modelo                                           
EQUATORIAL Atlas (fine-tuning)    98.77  0.84  98.49  1.03  
           Atlas (linear probe)   96.48  0.92  95.66  1.10  
           DINOv2 (fine-tuning)   98.37  1.40  97.96  1.70  
           DINOv2 (linear probe)  96.47  1.49  95.66  1.74  
           DenseNet201 (tese)     98.98  1.14  98.70  1.44  
POLAR      Atlas (fine-tuning)    99.38  0.48  99.21  0.60  
           Atlas (linear probe)   97.51  1.12  96.75  1.46  
           DINOv2 (fine-tuning)   98.98  0.97  98.66  1.24  
           DINOv2 (linear probe)  98.03  1.07  97.35  1.30  
           DenseNet201 (tese)     99.13  0.93  98.94  1.01  
TODAS      Atlas (fine-tuning)    99.05  0.57  98.95  0.64  
           Atlas (linear probe)   96.91  0.81  96.56  0.91  
           DINOv2 (fine-tuning)   98.62  0.97  98.47  1.07  
           DINOv2 (linear probe)  97.14  1.06  96.81  1.19

## 5. Teste de Wilcoxon pareado por fold (precisa dos 10 folds)

In [24]:
from scipy.stats import wilcoxon
def comparar(a_nome, b_nome, vista):
    a = tudo[(tudo.Modelo == a_nome) & (tudo.Vista == vista)].set_index("k").accuracy
    b = tudo[(tudo.Modelo == b_nome) & (tudo.Vista == vista)].set_index("k").accuracy
    ks = sorted(set(a.index) & set(b.index))
    if len(ks) < 10:
        print(f"{a_nome} x {b_nome} ({vista}): só {len(ks)} folds — rode os 10"); return
    dif = a[ks].values - b[ks].values
    try:
        p = wilcoxon(a[ks].values, b[ks].values, zero_method="zsplit").pvalue
        print(f"{a_nome} x {b_nome} ({vista}): diferença média {dif.mean()*100:+.2f} pp | p = {p:.4f}"
              f" -> {'significativo' if p < 0.05 else 'não significativo'}")
    except ValueError as e:
        print(f"{a_nome} x {b_nome} ({vista}): não foi possível calcular ({e})")

for nome in MODELOS_FT:
    for vista in ["EQUATORIAL", "POLAR"]:
        comparar(f"{nome} (fine-tuning)", "DenseNet201 (tese)", vista)
        comparar(f"{nome} (fine-tuning)", f"{nome} (linear probe)", vista)

Atlas (fine-tuning) x DenseNet201 (tese) (EQUATORIAL): diferença média -0.18 pp | p = 0.5898 -> não significativo
Atlas (fine-tuning) x Atlas (linear probe) (EQUATORIAL): diferença média +2.33 pp | p = 0.0020 -> significativo
Atlas (fine-tuning) x DenseNet201 (tese) (POLAR): diferença média +0.20 pp | p = 0.6602 -> não significativo
Atlas (fine-tuning) x Atlas (linear probe) (POLAR): diferença média +1.93 pp | p = 0.0039 -> significativo
DINOv2 (fine-tuning) x DenseNet201 (tese) (EQUATORIAL): diferença média -0.61 pp | p = 0.1562 -> não significativo
DINOv2 (fine-tuning) x DINOv2 (linear probe) (EQUATORIAL): diferença média +1.89 pp | p = 0.0098 -> significativo
DINOv2 (fine-tuning) x DenseNet201 (tese) (POLAR): diferença média -0.23 pp | p = 0.0547 -> não significativo
DINOv2 (fine-tuning) x DINOv2 (linear probe) (POLAR): diferença média +1.03 pp | p = 0.0156 -> significativo


In [25]:
for vista in ["EQUATORIAL", "POLAR"]:
    comparar("Atlas (fine-tuning)", "DINOv2 (fine-tuning)", vista)
display(resumo)

Atlas (fine-tuning) x DINOv2 (fine-tuning) (EQUATORIAL): diferença média +0.43 pp | p = 0.4609 -> não significativo
Atlas (fine-tuning) x DINOv2 (fine-tuning) (POLAR): diferença média +0.43 pp | p = 0.0312 -> significativo


accuracy       precision       recall        \
                                     mean   std      mean   std   mean   std   
Vista      Modelo                                                              
EQUATORIAL Atlas (fine-tuning)      98.76  0.85     98.95  0.81  98.76  0.85   
           Atlas (linear probe)     96.44  0.89     96.91  0.90  96.44  0.89   
           DINOv2 (fine-tuning)     98.33  1.39     98.70  1.25  98.33  1.39   
           DINOv2 (linear probe)    96.44  1.42     96.92  1.45  96.44  1.42   
           DenseNet201 (tese)       98.94  1.17     99.15  0.94  98.94  1.17   
POLAR      Atlas (fine-tuning)      99.38  0.48     99.44  0.48  99.38  0.48   
           Atlas (linear probe)     97.45  1.14     97.85  1.09  97.45  1.14   
           DINOv2 (fine-tuning)     98.95  0.97     99.11  0.84  98.95  0.97   
           DINOv2 (linear probe)    97.92  1.03     98.36  1.06  97.92  1.03   
           DenseNet201 (tese)       99.18  0.79     99.24  0.86  99.18  0.79   
TODAS      Atlas (fine-tuning)      99.06  0.57     99.11  0.55  99.06  0.57   
           Atlas (linear probe)     96.92  0.82     97.08  0.76  96.92  0.82   
           DINOv2 (fine-tuning)     98.63  0.96     98.75  0.85  98.63  0.96   
           DINOv2 (linear probe)    97.14  1.06     97.31  1.02  97.14  1.06   

                                 fscore        kappa        
                                   mean   std   mean   std  
Vista      Modelo                                           
EQUATORIAL Atlas (fine-tuning)    98.77  0.84  98.49  1.03  
           Atlas (linear probe)   96.48  0.92  95.66  1.10  
           DINOv2 (fine-tuning)   98.37  1.40  97.96  1.70  
           DINOv2 (linear probe)  96.47  1.49  95.66  1.74  
           DenseNet201 (tese)     98.98  1.14  98.70  1.44  
POLAR      Atlas (fine-tuning)    99.38  0.48  99.21  0.60  
           Atlas (linear probe)   97.51  1.12  96.75  1.46  
           DINOv2 (fine-tuning)   98.98  0.97  98.66  1.24  
           DINOv2 (linear probe)  98.03  1.07  97.35  1.30  
           DenseNet201 (tese)     99.13  0.93  98.94  1.01  
TODAS      Atlas (fine-tuning)    99.05  0.57  98.95  0.64  
           Atlas (linear probe)   96.91  0.81  96.56  0.91  
           DINOv2 (fine-tuning)   98.62  0.97  98.47  1.07  
           DINOv2 (linear probe)  97.14  1.06  96.81  1.19

In [26]:
for vista in ["EQUATORIAL", "POLAR"]:
    comparar("Atlas (fine-tuning)", "DINOv2 (fine-tuning)", vista)

Atlas (fine-tuning) x DINOv2 (fine-tuning) (EQUATORIAL): diferença média +0.43 pp | p = 0.4609 -> não significativo
Atlas (fine-tuning) x DINOv2 (fine-tuning) (POLAR): diferença média +0.43 pp | p = 0.0312 -> significativo
